# Clean measles notes
This script pulls all measles immunization dates from the "Immunizations" section of patient notes. These are then put into datasets in '/share/pi/deho/AFC/mortonc/intermediate/measles_notes', whose names start with 'immunizations_section'. These are compiled into '/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf.csv.zip'

This script also pulls all measles immunizations recorded in a visit from the "Procedures," "Procedures Ordered," and/or "Orders" sections. These are put into datasets in '/share/pi/deho/AFC/mortonc/intermediate/measles_notes', whose names start with 'procedures'. These are compiled into '/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf_proc.csv.zip'

Finally, this script pulls all measles immunizations recorded through having a note recording only "MMR," "MMR given," etc. These are compiled into '/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_non_rtf.csv.zip'.

This script then compiles all of these types of recorded measles immunizations (plus code/id from '/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax.csv.zip'), restricting to immunizations recorded over a year after patients were born only. It finds the earliest recorded measles immunization after the patient turned 1 from any source and saves it in  '/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_and_code.csv'

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from striprtf.striprtf import rtf_to_text
import re
from tqdm import tqdm
#tqdm.pandas() 
import swifter
from collections import defaultdict
import ast
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import gzip
import datetime

Idea: 
- First subset out all of the rtf from the non-rtf

- Figure out how many notes have MMR in only the immunization section

In [ ]:
def mmr_measles_only_in_immunizations(note_text):
    # Find where the "Immunizations:" section starts
    immun_start = note_text.find("Immunizations:")
    if immun_start == -1:
        return [False, []]  # No immunization section at all -> no dates
    
    # Find the next colon after the start of "Immunizations:"
    next_colon = note_text.find(":", immun_start + len("immunizations:"))
    if next_colon == -1:
        # If no colon after, assume immunization section goes to the end
        immun_block = note_text[immun_start:]
    else:
        # Get the section only from Immunizations: to the next colon
        immun_block = note_text[immun_start:next_colon]

    # Compile regex for date
    date_pattern = re.compile(r'(0?[1-9]|1[0-2])/(0?[1-9]|[12][0-9]|3[01])/\d{4}')

    # One-pass scan
    results = []
    i = 0
    waiting_for_date = False

    while i < len(immun_block):
        # Check for 'MMR' or 'measles' at current position (case-insensitive)
        if immun_block[i:i+3].lower() == 'mmr' and (i == 0 or not immun_block[i-1].isalnum()):
            waiting_for_date = True
            i += 3
            continue
        elif immun_block[i:i+7].lower() == 'measles' and (i == 0 or not immun_block[i-1].isalnum()):
            waiting_for_date = True
            i += 7
            continue

        # If waiting for date, check if a date starts here
        if waiting_for_date:
            match = date_pattern.match(immun_block, i)
            if match:
                results.append(match.group())
                waiting_for_date = False
                i += len(match.group())
                continue

        i += 1
        
    # Now check if "mmr" and "measles" appear outside the immun_block
    for keyword in ["mmr", "measles", "MMR", "mMR", "Measles"]:
        all_indices = [i for i in range(len(note_text)) if note_text.startswith(keyword, i)]
        for index in all_indices:
            if not (immun_start <= index < immun_start + len(immun_block)):
                return [False, results]  # Found outside the immunization section

    return [True, results]

def substring_before_period(text):
    period_index = text.find('.')
    if period_index == -1:
        return text
    else:
        return text[:period_index]
    
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

# For RTF files, extract "Immunizations" section

In [ ]:
path = '/share/pi/deho/AFC/BQ/PatientNote_0125_measles/'
filenames = os.listdir(path)

In [ ]:
# figure out our subset list of filenames we need to pull
files_already_made = os.listdir('/share/pi/deho/AFC/mortonc/intermediate/measles_notes/')
files_already_made = pd.DataFrame(files_already_made)[[x.endswith('csv.gz.csv') for x in files_already_made]][0]
files_already_made = [x[len('immunizations_section_'):-4] for x in files_already_made]

In [ ]:
filenames = set(filenames).difference(set(files_already_made))

In [ ]:

def process_file(filename, path, output_dir):
    try:
        # load note
        measles_note = pd.read_csv(path + filename)
        measles_note = measles_note.reindex()
        measles_note = measles_note[['patientuid', 'encounterdate', 'note']]

        # we are only looking at the rtf elements
        measles_note_rtf = measles_note[[str(x).lower().startswith('{\\\\rtf') for x in measles_note['note']]]

        # get measles dates
        found_mmr = [mmr_measles_only_in_immunizations(x) for x in measles_note_rtf['note']]
        measles_note_rtf['mmr_only_history'] = [x[0] for x in found_mmr]
        measles_note_rtf['mmr_dates'] = [x[1] for x in found_mmr]
        measles_note_rtf = measles_note_rtf[['patientuid', 'encounterdate', 'mmr_only_history', 'mmr_dates']]

        save_name = f"immunizations_section_{filename}.csv"
        output_path = os.path.join('/share/pi/deho/AFC/mortonc/intermediate/measles_notes/', save_name)
        measles_note_rtf.to_csv(output_path, index=False, compression="gzip")
        print(f"Processed: {filename}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Set paths
output_dir = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'

# Partial function to include constant arguments
process_file_partial = partial(process_file, path=path, output_dir=output_dir)

# Run in parallel
with ProcessPoolExecutor() as executor:
    executor.map(process_file_partial, filenames)

# Load and compile measles rtf dates from immunization section

In [ ]:
path = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'
filenames = os.listdir(path)
filenames = pd.DataFrame(filenames)[[x.startswith('imm') for x in filenames]][0]

In [ ]:
joined_data = pd.DataFrame(columns = ['patientuid', 'encounterdate', 'mmr_only_history', 'mmr_dates'])

In [ ]:
for filename in filenames:
    file = pd.read_csv(path + filename, compression = 'gzip')
    joined_data = pd.concat([joined_data, file])


In [ ]:
# filter to dates only on or after the patient was 1 year old
joined_data_filtered = joined_data[joined_data['mmr_dates'] != '[]'] # mmr dates is nonempty
joined_data_filtered = joined_data_filtered[['patientuid', 'mmr_dates']] # only dates and id

In [ ]:
# add birthdate column
with gzip.open('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz') as f:
    patient_baseline = pd.read_csv(f, low_memory = False)

In [ ]:
patient_baseline = patient_baseline[['patientuid', 'dob']]

In [ ]:
joined_data_filtered = pd.merge(joined_data_filtered, patient_baseline)

# find earliest date for each unique patient id
joined_data_filtered = joined_data_filtered.drop_duplicates()
joined_data_filtered = joined_data_filtered.reset_index()
joined_data_filtered['mmr_dates'] = [ast.literal_eval(x) for x in joined_data_filtered['mmr_dates']]

In [ ]:
joined_data_filtered = joined_data_filtered[np.logical_not(pd.isna(joined_data_filtered['dob']))]

In [ ]:
def get_earliest_valid_mmr(mmr_list, dob_str):
    dob = datetime.datetime.strptime(str(dob_str), '%Y-%m-%dT%H:%M:%SZ')
    one_year_later = dob + datetime.timedelta(days=365)  # simple 1-year buffer
    valid_dates = []
    for date_str in mmr_list:
        try:
            date_obj = datetime.datetime.strptime(date_str, '%m/%d/%Y')
            if date_obj >= one_year_later:
                valid_dates.append(date_obj)
        except Exception as e:
            pass  # Skip invalid date formats
    return min(valid_dates).strftime('%Y-%m-%d') if valid_dates else None

joined_data_filtered['earliest_valid_mmr'] = joined_data_filtered.apply(lambda row: get_earliest_valid_mmr(row['mmr_dates'], row['dob']), axis=1)

In [ ]:
joined_data_filtered = joined_data_filtered[['patientuid', 'earliest_valid_mmr']]

In [ ]:
joined_data_filtered = joined_data_filtered.rename(columns = {'earliest_valid_mmr': 'note_mmr_date'})

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf.csv', joined_data_filtered)

# Get notes that record giving mmr in procedures ordered

In [ ]:
def mmr_measles_only_in_proc(note_text):
    # Find where the "Immunizations:" section starts
    immun_start = note_text.lower().find("procedures ordered:")
    
    if immun_start == -1:
        immun_start = note_text.lower().find("procedures:")
        if immun_start == -1:
            immun_start = note_text.lower().find("orders:")
            if immun_start == -1:
                return False  # No immunization section at all -> no dates
    
    # Find the next colon after the start of "procedures:" or "procedures ordered:"
    next_colon = note_text.find(":", immun_start + len("procedures ordered:"))
    if next_colon == -1:
        # If no colon after, assume immunization section goes to the end
        immun_block = note_text[immun_start:]
    else:
        # Get the section only from Immunizations: to the next colon
        immun_block = note_text[immun_start:next_colon]

    # Now check if "mmr" and "measles" appear outside the immun_block
    for keyword in ["mmr", "measles", "MMR", "mMR", "Measles"]:
        all_indices = [i for i in range(len(immun_block)) if immun_block.startswith(keyword, i)]
        if len(all_indices) >= 1:
            return True

    return False


In [ ]:
def eval_measles_proc_file(filename, path, output_dir):
    try:
        # load note
        measles_note = pd.read_csv(path + filename)
        measles_note = measles_note.reindex()
        measles_note = measles_note[['patientuid', 'encounterdate', 'note']]
        # we are only looking at the rtf elements
        measles_note_rtf = measles_note[[str(x).lower().startswith('{\\\\rtf') for x in measles_note['note']]]

        summary_found = measles_note_rtf[[mmr_measles_only_in_proc(measles_note_rtf['note'][i]) for i in measles_note_rtf.index]]

        # save summary_found
        save_name = f"procedure_measles_{filename}.csv"
        output_path = os.path.join(output_dir, save_name)
        summary_found.to_csv(output_path, index=False, compression="gzip")
        print(f"Processed: {filename}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [ ]:
path = '/share/pi/deho/AFC/BQ/PatientNote_0125_measles/'
filenames = os.listdir(path)

# Set paths
output_dir = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'

# Partial function to include constant arguments
process_file_partial = partial(eval_measles_proc_file, path=path, output_dir=output_dir)

# Run in parallel
with ProcessPoolExecutor() as executor:
    executor.map(process_file_partial, filenames)

# Load and compile measles procedure dates

In [ ]:
# load files
path = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'
filenames = os.listdir(path)
filenames = pd.DataFrame(filenames)[[x.startswith('procedure_measles') for x in filenames]][0]


In [ ]:
all_proc_datasets = pd.DataFrame({'patientuid':[], 'encounterdate':[]})

for filename in filenames:
    try:
        if os.path.getsize(path + filename) != 0: # some notes (especially in earlier years) have 
                                                 # no rtf notes, so their "procedure" csvs are empty
            data = pd.read_csv(path + filename, compression='gzip')
            data = data[['patientuid', 'encounterdate']]
            all_proc_datasets = pd.concat([all_proc_datasets, data])
    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [ ]:
# restrict to dates only after the patient was 1 year old
def get_earliest_valid_mmr(mmr_date, dob_str):
    dob = datetime.datetime.strptime(str(dob_str), '%Y-%m-%dT%H:%M:%SZ')
    one_year_later = dob + datetime.timedelta(days=365)  # simple 1-year buffer
    
    try:
        date_obj = datetime.datetime.strptime(mmr_date, "%Y-%m-%d %H:%M:%S%z")
        date_obj = date_obj.replace(tzinfo=None)
        if date_obj >= one_year_later:
            return date_obj.strftime('%Y-%m-%d')
    except Exception as e:
            pass  # Skip invalid date formats
        


In [ ]:
joined_data_filtered = pd.merge(all_proc_datasets, patient_baseline)


In [ ]:
joined_data_filtered = joined_data_filtered.drop_duplicates()
joined_data_filtered = joined_data_filtered.reset_index()


In [ ]:
joined_data_filtered = joined_data_filtered[np.logical_not(pd.isna(joined_data_filtered['dob']))]

In [ ]:
joined_data_filtered['encounterdate'] = joined_data_filtered.apply(lambda row: get_earliest_valid_mmr(row['encounterdate'], row['dob']), axis=1)

In [ ]:
joined_data_filtered = joined_data_filtered[~pd.isna(joined_data_filtered['encounterdate'])]

In [ ]:
joined_data_filtered = joined_data_filtered[['patientuid', 'encounterdate']]

In [ ]:
joined_data_filtered

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf_proc.csv', joined_data_filtered)

# Get non-rtf dates

In [ ]:
path = '/share/pi/deho/AFC/BQ/PatientNote_0125_measles/'
filenames = os.listdir(path)

In [ ]:
def process_file(filename, path, output_dir):
    try:
        measles_note = pd.read_csv(path + filename)
        measles_note = measles_note.reindex()
        measles_note = measles_note[['patientuid', 'encounterdate', 'note']]

        # we are only looking at the non-rtf elements
        measles_note_no_rtf = measles_note[[np.logical_not(str(x).lower().startswith('{\\\\rtf')) for x in measles_note['note']]]

        # we want the short notes, ones that are just 'mmr' or have 'proquad' but aren't notes about them adding info to an existing vaccination etc.
        short_measles_note_no_rtf = measles_note_no_rtf[[len(str(x)) < 50 for x in measles_note_no_rtf['note']]]

        exclude_terms = [
            'titer', 'imm', 'hemmroid', 'hemmrroid', 'h/o', 'need', 'refus', 
            'overdue', 'status', 'state', '?', 'expir', 'updated', 'good', 'added'
        ]

        # Function to check if a note contains any of the exclude terms
        def contains_exclude_terms(note):
            note_lower = str(note).lower()
            return any(term in note_lower for term in exclude_terms)

        # Filter the DataFrame
        short_measles_note_no_rtf = short_measles_note_no_rtf[
            ~short_measles_note_no_rtf['note'].apply(contains_exclude_terms)
        ]

        # note should be MMR or MMRV or contain proquad + mmr/mmrv or start with mmr vacc or mmrv vac
        short_measles_note_no_rtf = short_measles_note_no_rtf[[str(x).lower() == 'mmr' or 
                                                               str(x).lower() == 'mmrv' or
                                                               (('proquad' in str(x).lower()) and (('mmr' or 'mmrv') in str(x).lower())) or
                                                                str(x).lower().startswith('mmr vacc') or 
                                                                str(x).lower().startswith('mmrv vacc')
                                                                 for x in short_measles_note_no_rtf['note']]]
        
        save_name = f"overall_note_{filename}.csv"
        output_path = os.path.join('/share/pi/deho/AFC/mortonc/intermediate/measles_notes/', save_name)
        short_measles_note_no_rtf.to_csv(output_path, index=False, compression="gzip")
        print(f"Processed: {filename}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Set paths
output_dir = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'

# Partial function to include constant arguments
process_file_partial = partial(process_file, path=path, output_dir=output_dir)

# Run in parallel
with ProcessPoolExecutor() as executor:
    executor.map(process_file_partial, filenames)

# Load and compile measles non-rtf dates

In [ ]:
path = '/share/pi/deho/AFC/mortonc/intermediate/measles_notes/'
filenames = os.listdir(path)
filenames = pd.DataFrame(filenames)[[x.startswith('over') for x in filenames]][0]

In [ ]:
joined_data = pd.DataFrame(columns = ['patientuid', 'encounterdate', 'note'])

In [ ]:
for filename in filenames:
    file = pd.read_csv(path + filename, compression = 'gzip')
    joined_data = pd.concat([joined_data, file])


In [ ]:
mmr_vals = joined_data['note'].value_counts()[joined_data['note'].value_counts() > 10].keys()

In [ ]:
joined_data_filtered = joined_data[joined_data['note'].isin(mmr_vals)]

In [ ]:
len(joined_data_filtered)

In [ ]:
joined_data_filtered = joined_data_filtered[['patientuid', 'encounterdate']] # only dates and id
joined_data_filtered = pd.merge(joined_data_filtered, patient_baseline)

# find earliest date for each unique patient id
joined_data_filtered = joined_data_filtered[np.logical_not(pd.isna(joined_data_filtered['dob']))]
joined_data_filtered = joined_data_filtered.drop_duplicates()
joined_data_filtered = joined_data_filtered.reset_index()


In [ ]:
len(joined_data_filtered)

In [ ]:
joined_data_filtered['one_year_from_dob']=[datetime.datetime.strptime(x, '%Y-%m-%dT%H:%M:%SZ') +\
                                             datetime.timedelta(days=365) for x in joined_data_filtered['dob']]

joined_data_filtered['one_year_from_dob'] = pd.to_datetime(joined_data_filtered['one_year_from_dob'])
joined_data_filtered['encounterdate'] = pd.to_datetime(joined_data_filtered['encounterdate'])
joined_data_filtered['encounterdate'] = joined_data_filtered['encounterdate'].dt.tz_localize(None)

# Filter rows where 'one_year_from_dob' is before 'encounterdate'
joined_data_filtered = joined_data_filtered[joined_data_filtered['one_year_from_dob'] <= joined_data_filtered['encounterdate']]

In [ ]:
len(joined_data_filtered)

In [ ]:
joined_data_filtered = joined_data_filtered[['patientuid', 'encounterdate']]
joined_data_filtered['encounterdate'] = pd.to_datetime(joined_data_filtered['encounterdate'])
joined_data_filtered = joined_data_filtered.drop_duplicates()
earliest_encounter = joined_data_filtered.groupby('patientuid', as_index=False)['encounterdate'].min()
earliest_encounter['encounterdate'] = earliest_encounter['encounterdate'].dt.strftime('%m-%d-%Y')


In [ ]:
earliest_encounter

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_non_rtf.csv', earliest_encounter)


# Compile all measles notes and non-notes vaccine dates

In [ ]:
non_rtf_measles = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_non_rtf.csv.zip')
rtf_measles_imm = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf.csv.zip')
rtf_measles_proc = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_rtf_proc.csv.zip')
code_measles = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax.csv.zip')

In [ ]:
rtf_measles_imm.columns = ['patientuid', 'date']
rtf_measles_proc.columns = ['patientuid', 'date']
non_rtf_measles.columns = ['patientuid', 'date']

In [ ]:
rtf_measles_imm['date'] = pd.to_datetime(rtf_measles_imm['date'], format = 'mixed', errors='coerce')
rtf_measles_proc['date'] = pd.to_datetime(rtf_measles_proc['date'], format = 'mixed', errors='coerce')

non_rtf_measles['date'] = pd.to_datetime(non_rtf_measles['date'], format = 'mixed', errors='coerce')
code_measles['date'] = pd.to_datetime(code_measles['date'], format = 'mixed', errors='coerce')

In [ ]:
# ensure code only includes dates after 1 year old
code_measles = pd.merge(code_measles, patient_baseline)

In [ ]:
code_measles['one_year_from_dob']=[datetime.datetime.strptime(x, '%Y-%m-%dT%H:%M:%SZ') +\
                                             datetime.timedelta(days=365) for x in code_measles['dob']]

code_measles['one_year_from_dob'] = pd.to_datetime(code_measles['one_year_from_dob'])
code_measles['date'] = pd.to_datetime(code_measles['date'])
code_measles['date'] = code_measles['date'].dt.tz_localize(None)

# Filter rows where 'one_year_from_dob' is before 'date'
code_measles = code_measles[code_measles['one_year_from_dob'] <= code_measles['date']]

In [ ]:
code_measles['date'] = pd.to_datetime(code_measles['date'], format = 'mixed', errors='coerce')
code_measles = code_measles[['patientuid', 'date']]

In [ ]:
measles_all = pd.concat([non_rtf_measles, rtf_measles_imm, rtf_measles_proc, code_measles]).drop_duplicates()

In [ ]:
earliest_measles_all = measles_all.groupby('patientuid', as_index=False)['date'].min()

In [ ]:
len(earliest_measles_all['patientuid'].unique()), len(earliest_measles_all)

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_and_code.csv', earliest_measles_all)
